# CellPert — End-to-End Tutorial

This notebook walks through a full CellPert use case from a clean checkout:

1. **Environment & data setup** — install dependencies and pull the LINCS / Tahoe datasets and pretrained weights from HuggingFace.
2. **Load the pretrained GINVAE model.**
3. **Build the reference biological context** from LINCS (control vs. perturb latent centroids).
4. **Predict perturbation responses** on a Tahoe query plate.
5. **Evaluate** the prediction (Pearson / Spearman / DEG-delta) on paired control–perturb samples.
All paths are relative to the repo root (`CellPert/`). After downloading the data the layout should look like:

```
CellPert/
├── src/
│   └── best_gin_vae_model_node_level.pth
├── data/
│   ├── lincs/merged_all_965_with_morgan.h5ad
│   └── minitahoe/p1_with_morgan.h5ad ... p14_with_morgan.h5ad
```

## 1. Environment & data

Run these once in a shell — *not* inside the notebook unless you want to (un)comment the `!`-prefixed lines.

```bash
conda env create -f environment.yml
conda activate info
```

Then download the model weights and datasets from HuggingFace (the repo is public, no token needed):

In [1]:
# One-time download — skip if you already have the files locally.
import os, zipfile, tarfile
from huggingface_hub import hf_hub_download

REPO_ID = 'Mike2481/CellPert'

os.makedirs('./data', exist_ok=True)

# Pretrained checkpoint
hf_hub_download(
    repo_id=REPO_ID, filename='best_gin_vae_model_node_level.pth',
    repo_type='dataset', local_dir='./src',
)

# LINCS reference
lincs_zip = hf_hub_download(
    repo_id=REPO_ID, filename='lincs.zip',
    repo_type='dataset', local_dir='./data',
)
with zipfile.ZipFile(lincs_zip) as z:
    z.extractall('./data/')

# Tahoe query plates (tar.gz)
tahoe_tar = hf_hub_download(
    repo_id=REPO_ID, filename='minitahoe.tar.gz',
    repo_type='dataset', local_dir='./data',
)
with tarfile.open(tahoe_tar, 'r:gz') as t:
    t.extractall('./data/')

lincs.zip:   0%|          | 0.00/1.57G [00:00<?, ?B/s]

minitahoe.tar.gz:   0%|          | 0.00/341M [00:00<?, ?B/s]

## 2. Load the pretrained GINVAE

`GINVAE` defaults: `input_dim=1, hidden_dim=300, latent_dim=100, num_layers=2`. Match these to the checkpoint.

In [2]:
import sys
sys.path.insert(0, './src')

import torch
from model import GINVAE

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GINVAE(input_dim=1, hidden_dim=300, latent_dim=100, num_layers=2).to(device)
model.load_state_dict(torch.load('./src/best_gin_vae_model_node_level.pth', map_location=device))
model.eval();

/blue/qsong1/wang.qing/miniconda3/envs/info/lib/python3.11/site-packages/torch_geometric/typing.py:97: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: Could not load this library: /blue/qsong1/wang.qing/miniconda3/envs/info/lib/python3.11/site-packages/torch_cluster/_version_cuda.so
  warnings.warn(f"An issue occurred while importing 'torch-cluster'. "


## 3. Build the reference biological context

CellPert transfers a perturbation effect from a *reference* dataset (LINCS) to a *query* dataset (Tahoe). The reference contributes two centroids in latent space:

$$z^{\text{perturb}}_{\text{query}} = z^{\text{ctrl}}_{\text{query}} + (\bar z^{\text{perturb}}_{\text{ref}} - \bar z^{\text{ctrl}}_{\text{ref}})$$

The first call processes LINCS and caches `latent_ctrl_ref` / `latent_ptrb_ref` to disk; subsequent calls are instant.

In [3]:
from dataset import ChunkedGeneGraphDataset
import os

os.makedirs('./output', exist_ok=True)

lincs_path = './data/lincs/merged_all_965_with_morgan.h5ad'
lincs_dataset = ChunkedGeneGraphDataset(
    h5ad_paths=[lincs_path], split='train',
    chunk_size=10000, auto_build_graph=True,
    species=9606, required_score=700,
)

latent_ctrl_ref, latent_ptrb_ref, info = model.process_reference_dataset(
    x_ref_dataset=lincs_dataset,
    save_path='./output/reference_latents.pkl',
    force_reprocess=False,
)
print('ctrl latent shape :', latent_ctrl_ref.shape)
print('perturb latent shape:', latent_ptrb_ref.shape)

Found 16 chunks with 157138 total samples
Processing reference dataset...
Processing and saving reference dataset with 157138 samples...
Using batch size: 100, saving every 100 batches


Processing reference batches:   6%|▋         | 99/1572 [00:08<01:39, 14.82it/s, samples=10000-10099, processed=1e+4]            

Saved chunk with 10000 samples


Processing reference batches:  13%|█▎        | 199/1572 [00:15<01:33, 14.66it/s, samples=19900-19999, processed=19900]

Saved chunk with 10000 samples


...

Processing completed. Total samples processed: 157138
Reference data metadata saved to ./output/reference_latents.pkl
Data chunks saved in ./output/reference_latents_chunks
Total samples processed: 157138

Summary statistics:
  condition:
    control: 78569
    perturb: 78569
  celltype:
    1HAE: 646
    22RV1: 1230
    5637: 6
    A204: 1234
    A375: 7118
    A549: 10144
    AGS: 98
    AN3CA: 6
    ASC: 4954
    BC3C: 1152
    BEN: 1248
    BICR6: 174
    BJAB: 146
    BT20: 272
    BT474: 74
    C42: 10
    CAL29: 1250
    CD34: 352
    CJM: 1240
    COV434: 172
    CW2: 6
    DU145: 8
    DV90: 6
    ES2: 166
    G401: 178
    GI1: 1226
    GP2D: 168
    H1975: 20
    HA1E: 8010
    HBL1: 180
    HCC1588: 158
    HCC515: 5092
    HCC827: 4
    HCC95: 1218
    HCT116: 98
    HEC108: 1224
    HEC151: 188
    HEC1A: 1258
    HEC251: 1202
    HEC265: 1240
    HEK293: 1660
    HEK293T: 56
    HELA: 2642
    HEPG2: 2226
    HFL1: 628
    HIMG001: 30
    HIMG002: 30
    HL60: 92
    HME

Loading chunks: 100%|██████████| 16/16 [00:00<00:00, 22.24it/s, file=chunk_15.pkl]


Loaded and filtered 157138 samples from chunks
Computing biological context from 78569 control and 78569 perturb samples
Extracting graph-level latent representations...


Processing perturb samples: 100%|██████████| 78569/78569 [00:00<00:00, 100901.40it/s]


Computing mean representations...
References computed successfully.
Control ref shape: torch.Size([1, 100]), norm: 3.5154
perturb ref shape: torch.Size([1, 100]), norm: 3.5216
Biological context norm: 0.0070
ctrl latent shape : torch.Size([1, 100])
perturb latent shape: torch.Size([1, 100])


## 4. Predict perturbation responses on a query plate

In [4]:
from torch_geometric.data import DataLoader

tahoe_path = './data/minitahoe/p1_with_morgan.h5ad'
tahoe_dataset = ChunkedGeneGraphDataset(
    h5ad_paths=[tahoe_path], split='test',
    chunk_size=10000, auto_build_graph=True,
    species=9606, required_score=700,
)

loader = DataLoader(tahoe_dataset, batch_size=64, shuffle=False)
batch = next(iter(loader)).to(device)
predicted_exp, mask = model.predict(batch, latent_ctrl_ref, latent_ptrb_ref)
print('predicted expression shape:', predicted_exp.shape)

Found 4 chunks with 39546 total samples


/blue/qsong1/wang.qing/miniconda3/envs/info/lib/python3.11/site-packages/torch_geometric/deprecation.py:26: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  warnings.warn(out)


predicted expression shape: torch.Size([64, 965, 1])


## 5. Evaluate on paired control–perturb samples

Re-uses the same evaluation pipeline that `src/main.py` runs in `--test_flag` mode:

* Filters Tahoe by `condition ∈ {control, perturb}`.
* Pairs them on `(celltype, main_ptrb, sub_ptrb)`.
* Computes per-sample Pearson / Spearman on absolute expression and on the DEG delta `pred − ctrl`.

Two equivalent ways to run it:

In [5]:
# Option A — call the helper directly from main.py
import argparse
from main import evaluate_perturbation_prediction

args = argparse.Namespace(
    device=device, batch_size=64,
    output_dir='./output', test_dataset_id=1,
)
results = evaluate_perturbation_prediction(
    model, tahoe_dataset, latent_ctrl_ref, latent_ptrb_ref,
    args, max_test_pairs=200, compute_deg=True,
)
print('absolute :', results['absolute_metrics'])
print('DEG delta:', results['deg_metrics'])

Filtering test dataset by condition...
Filtering dataset for condition: control


Filtering control samples: 100%|██████████| 39546/39546 [00:01<00:00, 33906.51it/s]


Found 19773 samples with condition 'control'
Filtering dataset for condition: perturb


Filtering perturb samples: 100%|██████████| 39546/39546 [00:01<00:00, 21689.06it/s]


Found 19773 samples with condition 'perturb'
Finding paired samples...
Found 19773 paired samples
Num of Test perturb:19773
Randomly selected 200 pairs for testing
Testing perturbation prediction on 200 paired samples...


Predicting perturbations: 100%|██████████| 4/4 [00:00<00:00, 16.37it/s]


Processing prediction results...
Processed 200 prediction pairs

Perturbation Prediction Metrics (Control → Predicted Stimulated vs Ground Truth Stimulated):
MSE                  40.841943
MAE                   6.379467
R2                 -391.892986
Pearson               0.295809
Spearman              0.158749
CosineSimilarity      0.410848
JS_Divergence         0.304281
SSIM                  0.010916
TopK_Overlap          0.000900
total_samples       200.000000
dtype: float64

Computing DEG (delta difference) metrics...

DEG Prediction Metrics (Predicted Delta vs Ground Truth Delta):
MSE                   40.842637
MAE                    6.379589
R2                 -1135.822248
Pearson                0.174971
Spearman               0.178482
CosineSimilarity       0.025448
JS_Divergence          0.004370
SSIM                  -0.002785
TopK_Overlap           0.047450
total_samples        200.000000
dtype: float64

✓ Results appended to: ./output/all_plates_results.csv
✓ Predictions sa

In [ ]:
# Option B — full sweep from the shell (one CSV row per plate)
# bash src/run_all.sh
# After it finishes, results land in:
#   output/all_plates_results.csv
#   output/predictions/plate_*_predictions.pkl